# TrackNet: run3 sui NOSTRI video (Colab, GPU)

Questo notebook fa **solo** il fine-tuning deciso per run3: parte dai pesi del **badminton** (`TrackNet_best.pt`, quelli del programma) e li corregge con i fotogrammi **etichettati a mano dai nostri video** (camera bassa dietro la linea di fondo), con passo basso (`LR = 1e-5`) e `10` epoche. Poi dà il **voto** sugli stessi 410 fotogrammi del set di test (video mai usati per addestrare), accanto a badminton, run1 e run2.

Il notebook generale (`finetune_tracknet_colab.ipynb`) resta com'è; qui mancano lo sguardo veloce e il dataset TV, che per run3 non servono.

**Prima di aprirlo:** sul PC `prepara_dataset_nostro.py` ha messo su Drive `tracknet_finetune/dati_nostri.tar`. **Aspetta che Google Drive abbia finito di caricarlo** (icona di Drive senza frecce che girano): se il tar è ancora in caricamento, Colab vede la versione vecchia o niente. La cella 5 mostra data e dimensione del tar e quanti fotogrammi ci sono per video: devono essere quelli stampati sul PC.

**Come si usa:** Runtime → Cambia tipo di runtime → **GPU A100** (va anche la T4, più lenta), poi **Runtime → Esegui tutto**. Serve solo l'autorizzazione del Drive (cella 2).
- Se la **prova** (cella 6) non riesce, il notebook si ferma lì e l'addestramento non parte: mandami l'errore.
- L'**addestramento** (cella 7) con circa 3.000 fotogrammi dura indicativamente 15-25 minuti su A100 (di più su T4). I risultati vanno su Drive in `tracknet_finetune/run3/`.
- Il **voto** (cella 9) rifà la previsione solo per run3: quelle di badminton, run1 e run2 sono già su Drive e si riusano.
- La **lettura del voto** (cella 10) mette in colonna il criterio deciso prima dei numeri e lo stesso confronto **senza il video djokovic del test** (girato a Melbourne come due video di addestramento).

- La **cella 11** scollega il runtime da sola alla fine, così la GPU non consuma unità di calcolo a vuoto (si può lanciare e andare a dormire).

Se Colab si disconnette durante l'addestramento: riapri ed "Esegui tutto": la cella 7 riprende dall'ultima epoca salvata. Se run3 è già finito, la cella 7 lo salta e si arriva dritti al voto.

## 1. Controllo GPU

In [ ]:
!nvidia-smi -L

## 2. Collega Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Codice: il nostro da GitHub e quello originale di TrackNetV3

Il codice di addestramento è quello **originale** di TrackNetV3 (qaz812345/TrackNetV3, licenza MIT), fermo alla versione verificata. La cella controlla che il modello sia identico a `tracknet3/model.py` del programma: così i pesi nuovi si usano senza cambiare niente.

In [ ]:
import os, shutil, glob, subprocess, sys, json

REPO = "anrundo-2312/rf_coach_vision"
DRIVE_DIR = "/content/drive/MyDrive/rf_coach_vision"
WORK_DIR = "/content/rf_coach_vision"
FT_DRIVE = os.path.join(DRIVE_DIR, "tracknet_finetune")      # risultati del fine-tuning su Drive
TNV3_DIR = "/content/TrackNetV3"                               # codice originale di addestramento
TNV3_COMMIT = "6eda442ada1740573f200f836d93edc9a541ee86"       # versione verificata

# Il repo e' pubblico: il token serve solo se un giorno torna privato (come nel notebook principale).
token = ""
try:
    from google.colab import userdata
    token = (userdata.get("GITHUB_TOKEN") or "").strip()
except Exception:
    pass
url = f"https://{REPO.split('/')[0]}:{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

os.chdir("/content")
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
r = subprocess.run(["git", "clone", "--depth", "1", url, WORK_DIR], capture_output=True, text=True,
                   env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
if r.returncode != 0:
    raise SystemExit("Download da GitHub non riuscito:\n" + (r.stderr.replace(token, "***") if token else r.stderr))
assert os.path.isdir(os.path.join(WORK_DIR, "tracknet_finetune")), "Su GitHub manca tracknet_finetune/: fai prima il push dal PC"

if not os.path.isdir(TNV3_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/qaz812345/TrackNetV3.git", TNV3_DIR], check=True)
subprocess.run(["git", "-C", TNV3_DIR, "checkout", "-q", TNV3_COMMIT], check=True)

leggi = lambda p: open(p, encoding="utf-8").read().replace("\r\n", "\n")
assert leggi(os.path.join(TNV3_DIR, "model.py")) == leggi(os.path.join(WORK_DIR, "tracknet3", "model.py")), \
    "Il modello di TrackNetV3 non e' identico a tracknet3/model.py: non proseguire"
print("codice pronto; modello identico a quello del programma")

In [ ]:
!pip install -q parse pycocotools gdown

## 4. Pesi di partenza (quelli del badminton che usa il programma)

In [ ]:
PESI_INIZIALI = os.path.join(WORK_DIR, "tracknet3", "ckpts", "TrackNet_best.pt")
os.makedirs(os.path.dirname(PESI_INIZIALI), exist_ok=True)
shutil.copy2(os.path.join(DRIVE_DIR, "ckpts", "TrackNet_best.pt"), PESI_INIZIALI)   # copy2: tiene la data del file
print("pesi di partenza:", PESI_INIZIALI, f"({os.path.getsize(PESI_INIZIALI) / 1e6:.0f} MB)")

## 5. Dataset dai nostri video

Scompatta `tracknet_finetune/dati_nostri.tar` (fatto sul PC da `prepara_dataset_nostro.py`). Lo scompatta **ogni volta da capo**, così se il dataset è stato rifatto non resta in giro quello vecchio. Mostra data e dimensione del tar e i fotogrammi per video: confrontali con quello che ha stampato il PC.

In [ ]:
import collections, datetime

DATI_NOSTRI = "/content/dati_nostri"
tar_nostri = os.path.join(FT_DRIVE, "dati_nostri.tar")
if not os.path.exists(tar_nostri):
    raise SystemExit("Su Drive manca tracknet_finetune/dati_nostri.tar: sul PC lancia prepara_dataset_nostro.py "
                     "e aspetta che Google Drive finisca di caricarlo.")
st = os.stat(tar_nostri)
print(f"tar: {tar_nostri}")
print(f"  {st.st_size / 1e6:.0f} MB, modificato il "
      f"{datetime.datetime.fromtimestamp(st.st_mtime):%d/%m/%Y %H:%M} (ora UTC di Colab)")
shutil.rmtree(DATI_NOSTRI, ignore_errors=True)          # sempre da capo: niente resti di un dataset vecchio
subprocess.run(["tar", "-xf", tar_nostri, "-C", "/content"], check=True)
r = json.load(open(os.path.join(DATI_NOSTRI, "riepilogo_dataset.json")))
for split, s in r["split"].items():
    print(f"{split}: {s['fotogrammi']} fotogrammi in {len(s['clip'])} clip, pallina visibile {s['visibili']}, mossa {s['mosse']}")
print("scartati:", r.get("scartati"))

# fotogrammi per video (train + val), dai CSV delle clip
per_video = collections.Counter()
for split, s in r["split"].items():
    for m in sorted(glob.glob(os.path.join(DATI_NOSTRI, split, "match*"))):
        for c in glob.glob(os.path.join(m, "csv", "*_ball.csv")):
            nome = os.path.basename(c)[:-len("_ball.csv")]
            video = next((cl.split("/")[0] for cl in s["clip"] if cl.endswith("/" + nome)), "?")
            per_video[(video, split)] += sum(1 for _ in open(c)) - 1
print("\nfotogrammi per video (addestramento / validazione):")
for v in sorted({v for v, _ in per_video}):
    print(f"  {v}: {per_video[(v, 'train')]} / {per_video[(v, 'val')]}")
if not r["split"]["val"]["clip"]:
    raise SystemExit("Nessuna clip di validazione: addestra.py non potrebbe scegliere l'epoca migliore.")

## 6. Prova veloce (2-3 minuti)

256 campioni, 1 epoca, sui nostri dati: serve solo a scoprire subito se qualcosa non va (dati, memoria, versioni). Deve finire con `PROVA RIUSCITA`; se non riesce, la cella si ferma e con "Esegui tutto" l'addestramento non parte. Scrive in una cartella temporanea, non sul Drive.

In [ ]:
%cd {WORK_DIR}
shutil.rmtree("/content/prova_ft", ignore_errors=True)
!python tracknet_finetune/addestra.py --dati {DATI_NOSTRI} --tracknetv3 {TNV3_DIR} --pesi_iniziali {PESI_INIZIALI} --uscita /content/prova_ft --prova
if not os.path.exists("/content/prova_ft/ultimo_completo.pt"):
    raise SystemExit("La prova NON e' riuscita: copia l'errore qui sopra e mandamelo in chat.")
print("\nProva riuscita: l'addestramento (cella 7) puo' partire.")

## 7. Addestramento run3

Prima misura i pesi del badminton sulla validazione (un tratto ogni 4 dei nostri video, mai visto in addestramento), poi addestra per `EPOCHE` epoche e dopo ognuna rimisura: tiene la versione migliore. Tutto va su Drive in `tracknet_finetune/run3/`:
- `TrackNet_tennis.pt` (la migliore: è quella che si vota),
- `ultimo_completo.pt` (per riprendere),
- `registro.json` (i numeri di ogni epoca).

Impostazioni decise: dai pesi del badminton, `LR = 1e-5`, `10` epoche, niente congelato. La cella decide da sola: se run3 è già finito **salta**; se era stato interrotto **riprende**; altrimenti parte da zero.

In [ ]:
RUN = "run3"          # nome della prova (run1 e run2 = dataset TV, restano su Drive)
EPOCHE = 10
LR = 1e-5             # passo basso: pochi dati, si corregge la rete del badminton
CONGELA = "nessuno"   # "inizio" = tiene fermi i primi blocchi (se la validazione peggiora presto)
DATI_RUN = DATI_NOSTRI   # i nostri video (cella 5)

USCITA = os.path.join(FT_DRIVE, RUN)
p_reg = os.path.join(USCITA, "registro.json")
fatte = len(json.load(open(p_reg))["epoche"]) if os.path.exists(p_reg) else 0
if fatte >= EPOCHE and os.path.exists(os.path.join(USCITA, "TrackNet_tennis.pt")):
    print(f"{RUN}: addestramento gia' finito ({fatte} epoche), salto.")
else:
    opz_riprendi = ""
    if os.path.exists(os.path.join(USCITA, "ultimo_completo.pt")):
        opz_riprendi = "--riprendi"
        print(f"{RUN}: riprendo dall'epoca {fatte + 1}")
    %cd {WORK_DIR}
    !python tracknet_finetune/addestra.py --dati {DATI_RUN} --tracknetv3 {TNV3_DIR} --pesi_iniziali {PESI_INIZIALI} --uscita "{USCITA}" --epoche {EPOCHE} --lr {LR} --congela {CONGELA} {opz_riprendi}
fatte = len(json.load(open(p_reg))["epoche"]) if os.path.exists(p_reg) else 0
if fatte < EPOCHE or not os.path.exists(os.path.join(USCITA, "TrackNet_tennis.pt")):
    raise SystemExit(f"Addestramento INTERROTTO dopo {fatte} epoche su {EPOCHE} (lo stato e' salvato su Drive): "
                     "rilancia questa cella per riprendere. Se qui sopra c'e' un errore, mandamelo in chat.")
print(f"\nAddestramento finito ({fatte} epoche). Pesi nuovi:", os.path.join(USCITA, "TrackNet_tennis.pt"))

## 8. Andamento sulla validazione (tratteggio = pesi del badminton)

In [ ]:
import matplotlib.pyplot as plt
reg = json.load(open(os.path.join(USCITA, "registro.json")))
ep = [e["epoca"] for e in reg["epoche"]]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, nome in (("accuracy", "accuratezza"), ("f1", "F1"), ("recall", "richiamo")):
    ax[0].plot(ep, [e[k] for e in reg["epoche"]], marker="o", label=nome)
    if "prima" in reg:
        ax[0].axhline(reg["prima"][k], ls="--", lw=0.8, color=ax[0].lines[-1].get_color())
ax[0].set_title("validazione (tratteggio = pesi del badminton)"); ax[0].set_xlabel("epoca"); ax[0].legend()
ax[1].plot(ep, [e["perdita_train"] for e in reg["epoche"]], marker="o", label="addestramento")
ax[1].plot(ep, [e["perdita_val"] for e in reg["epoche"]], marker="o", label="validazione")
ax[1].set_title("perdita"); ax[1].set_xlabel("epoca"); ax[1].legend()
plt.show()
if "migliore" in reg:
    print("migliore: epoca", reg["migliore"]["epoca"], "accuratezza", round(reg["migliore"]["accuracy"], 3))

## 9. Voto sul set di test (i nostri video del test, etichettati a mano)

Confronta badminton, run1, run2 e run3 sugli **stessi 410 fotogrammi**, con la previsione fatta **come nel programma** (`tracknet3/predict.py`, senza InpaintNet). Le previsioni già fatte restano su Drive (`tracknet_finetune/valutazione/previsioni`) e vengono riusate: si calcola solo run3.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

# I pesi da confrontare sugli stessi fotogrammi. Se una cartella non c'e' ancora, viene saltata.
PESI = {"badminton": PESI_INIZIALI,
        "run1": os.path.join(FT_DRIVE, "run1", "TrackNet_tennis.pt"),
        "run2": os.path.join(FT_DRIVE, "run2", "TrackNet_tennis.pt"),
        "run3": os.path.join(FT_DRIVE, "run3", "TrackNet_tennis.pt")}

mancano = [n for n, p in PESI.items() if not os.path.exists(p)]
for n in mancano:
    print(f"(salto '{n}': pesi non ancora addestrati)")
    PESI.pop(n)

et_drive = os.path.join(FT_DRIVE, "etichette_test.csv")
et_repo = os.path.join(WORK_DIR, "tracknet_finetune", "test", "etichette_test.csv")
ETICHETTE = et_drive if os.path.exists(et_drive) else et_repo
if not os.path.exists(ETICHETTE):
    raise SystemExit("Set di test non ancora etichettato: lancia questa cella quando hai finito con etichetta_pallina.py sul PC.")
et = pd.read_csv(ETICHETTE)
print(f"set di test: {ETICHETTE}")
print(f"  {len(et)} etichette: visibile {(et['visibile'] == 1).sum()}, non visibile {(et['visibile'] == 0).sum()}, "
      f"incerta {(et['visibile'] == -1).sum()} (non conta)")

os.makedirs(os.path.join(WORK_DIR, "inputs"), exist_ok=True)
for v in sorted(et["video"].unique()):
    src, dst = os.path.join(DRIVE_DIR, "inputs", v), os.path.join(WORK_DIR, "inputs", v)
    if not os.path.exists(src):
        print("  manca su Drive inputs/:", v, "(le sue etichette vengono saltate)")
    elif not os.path.exists(dst):
        shutil.copy(src, dst)

VAL_LOCALE, VAL_DRIVE = "/content/valutazione", os.path.join(FT_DRIVE, "valutazione")
if os.path.isdir(os.path.join(VAL_DRIVE, "previsioni")):
    shutil.copytree(os.path.join(VAL_DRIVE, "previsioni"), os.path.join(VAL_LOCALE, "previsioni"), dirs_exist_ok=True)

pesi_arg = " ".join(f'"{n}={p}"' for n, p in PESI.items())
%cd {WORK_DIR}
!python tracknet_finetune/valuta_tracknet.py --etichette "{ETICHETTE}" --video_dir inputs --pesi {pesi_arg} --uscita {VAL_LOCALE}

os.makedirs(VAL_DRIVE, exist_ok=True)
for f in glob.glob(os.path.join(VAL_LOCALE, "voto.*")) + glob.glob(os.path.join(VAL_LOCALE, "confronto_*.csv")):
    shutil.copy(f, VAL_DRIVE)
shutil.copytree(os.path.join(VAL_LOCALE, "previsioni"), os.path.join(VAL_DRIVE, "previsioni"), dirs_exist_ok=True)
display(Markdown(open(os.path.join(VAL_LOCALE, "voto.md"), encoding="utf-8").read()))
print("Copia dei risultati su Drive:", VAL_DRIVE)

## 10. Lettura del voto

Dai file `confronto_<pesi>.csv` della cella 9, senza rifare previsioni:
1. **il criterio fissato prima dei numeri**: richiamo *vicino al colpo* almeno +8-10 punti rispetto al badminton e FP2 *a caso* non in aumento;
2. **trovata / visibili** e **punti sbagliati / visibili** vicino al colpo (più onesti del richiamo di TrackNetV3, che non conta i FP1 al denominatore);
3. gli stessi numeri **senza il video djokovic del test** (`zverev_djokovic_trim_swin_like`): è girato a Melbourne come `sinner_sparring1` e `alcaraz_ruud` dell'addestramento, quindi lì run3 potrebbe essere avvantaggiato.

In [ ]:
sys.path.insert(0, os.path.join(WORK_DIR, "tracknet_finetune"))
import valuta_tracknet as vt

VIDEO_MELBOURNE = "zverev_djokovic_trim_swin_like.mp4"     # video del test girato sullo stesso campo di due video di addestramento
conf = {n: pd.read_csv(os.path.join(VAL_LOCALE, f"confronto_{n}.csv")) for n in PESI
        if os.path.exists(os.path.join(VAL_LOCALE, f"confronto_{n}.csv"))}
fps_video = {v: vt.info_video(os.path.join(WORK_DIR, "inputs", v))[0] for v in et["video"].unique()
             if os.path.exists(os.path.join(WORK_DIR, "inputs", v))}

def tabella(filtro, titolo):
    righe = []
    for n, df in conf.items():
        g = vt.gruppi(df[filtro(df)], fps_video)
        vic, caso = g["vicino al colpo (0,1 s)"], g["a caso (lontano dai colpi)"]
        m, mc = vt.metriche(vic), vt.metriche(caso)
        visibili = max(m["TP"] + m["FP1"] + m["FN"], 1)
        righe.append({"pesi": n, "vicino: n": m["n"], "vicino: richiamo": round(m["richiamo"], 2),
                      "vicino: trovata/visibili": round(m["TP"] / visibili, 2),
                      "vicino: sbagliata/visibili": round(m["FP1"] / visibili, 2),
                      "vicino: F1": round(m["F1"], 2), "a caso: n": mc["n"], "a caso: FP2": mc["FP2"],
                      "tutti: richiamo": round(vt.metriche(g["tutti"])["richiamo"], 2)})
    t = pd.DataFrame(righe).set_index("pesi")
    display(Markdown(f"**{titolo}**"))
    display(t)
    return t

t_tutti = tabella(lambda d: d["video"] == d["video"], "Tutti i video del test")
t_senza = tabella(lambda d: d["video"] != VIDEO_MELBOURNE, f"Senza {VIDEO_MELBOURNE}")

if "run3" in t_tutti.index and "badminton" in t_tutti.index:
    for t, nome in ((t_tutti, "tutti i video"), (t_senza, "senza djokovic")):
        d_rich = 100 * (t.loc["run3", "vicino: richiamo"] - t.loc["badminton", "vicino: richiamo"])
        d_trov = 100 * (t.loc["run3", "vicino: trovata/visibili"] - t.loc["badminton", "vicino: trovata/visibili"])
        d_fp2 = t.loc["run3", "a caso: FP2"] - t.loc["badminton", "a caso: FP2"]
        ok = d_rich >= 8 and d_fp2 <= 0
        print(f"[{nome}] run3 rispetto al badminton, vicino al colpo: richiamo {d_rich:+.0f} punti, "
              f"trovata/visibili {d_trov:+.0f} punti; FP2 a caso {d_fp2:+d}  ->  criterio "
              f"{'RAGGIUNTO' if ok else 'NON raggiunto'} (serve richiamo +8-10 e FP2 non in aumento)")
    print("\nNumeri piccoli (circa 24 colpi veri): differenze di pochi punti non sono significative.")
else:
    print("run3 non c'e' ancora nel voto: prima la cella 7 e la cella 9.")

## 11. Scollega il runtime (ferma il consumo di unità)

Le unità di calcolo si consumano finché il notebook è collegato alla GPU, anche a lavoro finito. Questa cella scollega il runtime da sola: con "Esegui tutto" gira per ultima, quando tutto è andato bene. Non si perde niente: pesi, registro e voto sono già su Drive (`tracknet_finetune/run3/` e `valutazione/`) e gli output delle celle restano nel notebook. Se una cella prima si ferma con un errore, questa non gira e il runtime resta collegato finché Colab non lo scollega per inattività.

In [ ]:
print("Tutto finito: risultati su Drive in", USCITA, "e", VAL_DRIVE)
print("Scollego il runtime per non consumare unita' di calcolo a vuoto.")
from google.colab import runtime
runtime.unassign()

## Dopo

- Se run3 **raggiunge il criterio** (anche senza il video djokovic): integrazione nel programma, con il confronto A/B sui video di riferimento (LEGGIMI, "Dopo"). Prima il controllo incrociato con GPT e l'OK di Andrea.
- Se **non** lo raggiunge: guardare *trovata/visibili* e *sbagliata/visibili* e il grafico della cella 8 (sovradattamento presto o no) prima di decidere un'altra prova. Niente nuove prove senza piano condiviso.